# Вариант 1

In [1]:
import pandas as pd
import numpy as np

# Практическая работа 3
### Работа с реальными данными

**Цель:** Научиться работать с реальными файлами данных разных форматов, читать большие CSV по частям, выбирать только нужные столбцы, задавать типы данных и сохранять результаты.

### Исходные данные

Файл `data/orders_2024_01.csv` — лог заказов интернет-магазина за январь 2024 года.

Столбцы:
- `order_id` — идентификатор заказа
- `user_id` — идентификатор пользователя
- `order_datetime` — дата и время заказа
- `product_id` — идентификатор товара
- `price` — стоимость позиции
- `quantity` — количество единиц товара
- `status` — статус заказа (paid, cancelled, refunded и т.п.)

### Задание 1. Считать CSV orders_2024_01.csv по частям (chunksize=50_000), посчитать общую выручку (price * quantity)

In [2]:
csv_path = "data/orders_2024_01.csv"
chunk_size = 50_000  # количество строк в одном чанке

total_revenue = 0
rows_count = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    # вычисляем выручку в чанке
    chunk["revenue"] = chunk["price"] * chunk["quantity"]
    total_revenue += chunk["revenue"].sum()
    rows_count += len(chunk)

print("Общее количество строк:", rows_count)
print("Общая выручка за период:", total_revenue)

Общее количество строк: 5000
Общая выручка за период: 129378775.5


### Задание 2. Посчитать количество строк в файле через суммирование len(chunk) по всем чанкам

In [3]:
csv_path = "data/orders_2024_01.csv"
chunk_size = 50_000

total_rows = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    total_rows += len(chunk)

print("Количество строк в файле (через чанки):", total_rows)

Количество строк в файле (через чанки): 5000


### Задание 3. Считать только столбцы order_id, user_id, price, quantity с оптимизированными типами данных

In [4]:
csv_path = "data/orders_2024_01.csv"

use_columns = ["order_id", "user_id", "price", "quantity"]
dtypes = {
    "order_id": "int64",
    "user_id": "int32",
    "price": "float32",
    "quantity": "int16"
}

df_opt = pd.read_csv(
    csv_path,
    usecols=use_columns,
    dtype=dtypes
)

print(df_opt.info())
print(df_opt.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5000 non-null   int64  
 1   user_id   5000 non-null   int32  
 2   price     5000 non-null   float32
 3   quantity  5000 non-null   int16  
dtypes: float32(1), int16(1), int32(1), int64(1)
memory usage: 88.0 KB
None
   order_id  user_id     price  quantity
0         1     1044  14038.15         1
1         2     1079   9172.60         1
2         3     1041   9035.26         2
3         4     1083  15234.33         3
4         5     1068   4134.38         2


### Задание 4. Сравнить использование памяти при чтении CSV без параметров и с usecols/dtype

In [5]:
csv_path = "data/orders_2024_01.csv"

# Без оптимизации — читаем все столбцы с типами по умолчанию
df_full = pd.read_csv(csv_path)
print("=== Без оптимизации ===")
print(df_full.info(memory_usage="deep"))

print()

# С оптимизацией — только нужные столбцы и компактные типы
use_columns = ["order_id", "user_id", "price", "quantity"]
dtypes = {
    "order_id": "int64",
    "user_id": "int32",
    "price": "float32",
    "quantity": "int16"
}
df_opt = pd.read_csv(csv_path, usecols=use_columns, dtype=dtypes)
print("=== С оптимизацией ===")
print(df_opt.info(memory_usage="deep"))

print()
mem_full = df_full.memory_usage(deep=True).sum()
mem_opt  = df_opt.memory_usage(deep=True).sum()
print(f"Память без оптимизации : {mem_full / 1024:.1f} KB")
print(f"Память с оптимизацией  : {mem_opt  / 1024:.1f} KB")
print(f"Экономия               : {(1 - mem_opt / mem_full) * 100:.1f}%")

=== Без оптимизации ===
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        5000 non-null   int64  
 1   user_id         5000 non-null   int64  
 2   order_datetime  5000 non-null   str    
 3   product_id      5000 non-null   int64  
 4   price           5000 non-null   float64
 5   quantity        5000 non-null   int64  
 6   status          5000 non-null   str    
dtypes: float64(1), int64(4), str(2)
memory usage: 790.7 KB
None

=== С оптимизацией ===
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5000 non-null   int64  
 1   user_id   5000 non-null   int32  
 2   price     5000 non-null   float32
 3   quantity  5000 non-null   int16  
dtypes: float32(1), int16(1), int32(1), int64(1)
memory us

### Задание 5. Сохранить результат по выручке по пользователям (user_id → total_revenue) в новый файл user_revenue_2024_01.csv

In [6]:
csv_path = "data/orders_2024_01.csv"
chunk_size = 50_000

# Накапливаем выручку по пользователям чанк за чанком
user_revenue = {}

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    chunk["revenue"] = chunk["price"] * chunk["quantity"]
    chunk_user_rev = chunk.groupby("user_id")["revenue"].sum()
    for uid, rev in chunk_user_rev.items():
        user_revenue[uid] = user_revenue.get(uid, 0) + rev

# Формируем итоговый DataFrame
user_rev_df = pd.DataFrame(
    list(user_revenue.items()),
    columns=["user_id", "total_revenue"]
)
user_rev_df = user_rev_df.sort_values("user_id").reset_index(drop=True)

# Сохраняем в CSV
out_path = "data/user_revenue_2024_01.csv"
user_rev_df.to_csv(out_path, index=False)

print("Файл сохранён:", out_path)
print("Количество пользователей:", len(user_rev_df))
print()
print(user_rev_df.head(10))

Файл сохранён: data/user_revenue_2024_01.csv
Количество пользователей: 100

   user_id  total_revenue
0     1001     1679780.92
1     1002      756157.70
2     1003     1290829.44
3     1004     1373915.76
4     1005      802000.08
5     1006     1407863.62
6     1007     1168524.89
7     1008      927758.31
8     1009     1302862.51
9     1010     1589574.52
